In [ ]:
import serial.tools.list_ports
import plotly.graph_objects as go
import io
from time import sleep, time
from numpy import percentile, mean
from IPython.display import display, update_display, Image

BAUDRATE = 115200  # Doit correspondre à Serial.begin() dans le programme Arduino (optimisé)

def parse_ligne(ligne):
    """Retourne (temps_us, tension_V) si la ligne 'temps_us,tension' est valide, sinon None."""
    # errors='replace' : un octet parasite émis par la carte à l'ouverture du port
    # (ex. pendant le reset DTR) rend la ligne invalide sans faire planter la lecture
    ligne = ligne.decode('utf-8', errors='replace').strip() if isinstance(ligne, bytes) else ligne.strip()
    if not ligne or ',' not in ligne:
        return None
    try:
        temps, tension = ligne.split(',')
        return float(temps), float(tension)
    except ValueError:
        return None

def detect_arduino():
    ports = list(serial.tools.list_ports.comports())
    # 1) Recherche prioritaire d'une carte Arduino (fabricant Arduino, VID 2341)
    for port in ports:
        if "Arduino" in port.description or port.vid == 0x2341:
            return port.device
    # 2) En dernier recours, n'importe quel port série (ttyACM/ttyUSB)
    for port in ports:
        if "ttyACM" in port.device or "ttyUSB" in port.device:
            return port.device
    raise Exception("Arduino non détecté. Veuillez vérifier la connexion.")

In [ ]:
def connexion_arduino():
    global ser
    # Éviter de rouvrir le port s'il est déjà ouvert (relance de la cellule)
    if 'ser' in globals() and ser.is_open:
        print(f"Connexion déjà ouverte sur {ser.port}.")
        return
    # Détecter automatiquement le port série de l'Arduino
    port = detect_arduino()
    print(f"Arduino détecté sur {port}.")
    # Ouvrir la connexion série et attendre la première ligne valide
    ser = serial.Serial(port, BAUDRATE, timeout=2)
    lire_ligne_valide()

In [ ]:
def reset_arduino():
    ser.reset_input_buffer()
    # Envoyer la commande de reset
    ser.setDTR(False)    # DTR (Data Terminal Ready) à l'état bas pendant un court instant
    sleep(0.1)
    ser.reset_input_buffer()
    #ser.flushInput()    # reinitialisation des données
    ser.setDTR(True)    # DTR à l'état haut pour terminer le reset

In [ ]:
def mesure_unique():
    # L'Arduino émet en continu : on vide d'abord les données accumulées pendant la saisie
    while ser.in_waiting :
        ser.read(ser.in_waiting)
    tensions = []
    for _ in range(10) :
        parse = parse_ligne(ser.readline())
        if parse is not None:
            tensions.append(parse[1])
    if len(tensions) == 0 :
        raise Exception("Aucune mesure reçue de l'Arduino. Vérifier le câble USB et le programme acquisition_continue.ino.")
    ## on élimine les valeurs aberrantes par la méthode IQR :
    # Calculer le premier quartile (25ème percentile)
    Q1 = percentile(tensions, 25)
    
    # Calculer le troisième quartile (75ème percentile)
    Q3 = percentile(tensions, 75)
    
    # Calculer l'IQR
    IQR = Q3 - Q1
    
    # Définir les limites inférieure et supérieure pour les valeurs aberrantes
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Supprimer les valeurs aberrantes
    tensions_non_aberrantes = [value for value in tensions if lower_bound <= value <= upper_bound]

    return mean(tensions_non_aberrantes)

In [ ]:
def lire_ligne_valide():
    lectures_vides = 0
    while True:
        ligne = ser.readline()
        parse = parse_ligne(ligne)
        if parse is not None:
            return parse[0], ligne.decode('utf-8').strip()
        if ligne:
            lectures_vides = 0  # ligne reçue mais invalide : on continue
        else:
            lectures_vides += 1  # lecture vide : la carte n'envoie plus rien
            if lectures_vides > 10:
                raise Exception("Aucune donnée reçue de l'Arduino : vérifier le câble USB.")

In [ ]:
def acquisition_donnees(duree):
    """Enregistre les mesures (temps en s, tension en V) dans arduino.csv.

    La durée `duree` est en secondes ; l'horodatage série est en µs."""
    donnees = "t,U"
    
    while True :
        try:
            temps, ligne = lire_ligne_valide()
            temps_us, tension = temps, parse_ligne(ligne)[1]
            donnees += f"\n{temps_us*1e-6:.6f},{tension:.6f}"
            if temps_us >= duree * 1e6 :
                break
        except KeyboardInterrupt:
            # Interrompre la boucle si l'utilisateur appuie sur le bouton stop
            break
    print(donnees)
    # Fermer la connexion série
    fichier = open("arduino.csv", 'w')
    fichier.write(donnees)
    fichier.close()
    ser.close()

In [ ]:
def update_graph():
    global t, u, t_brut, u_brut, fig_graphique, temps_initial, last_update_time, interval, dernier_temps, affichage_temps_reel, duree_courante
    try:
        parse = parse_ligne(ser.readline())
        if parse is None:
            return False  # aucune donnée disponible

        temps_us, tension = parse
        temps = temps_us * 1e-6  # Conversion en secondes (µs → s)

        if temps_initial is None:
            temps_initial = temps

        temps_relatif = temps - temps_initial
        dernier_temps = temps_relatif

        # Conserver TOUS les échantillons (CSV plein débit)
        t_brut.append(temps_relatif)
        u_brut.append(tension)

        # Affichage : un point par intervalle en temps réel (~100 points max) ;
        # pour les durées très courtes (interval = 0) on collecte tout sans
        # toucher au canevas : le tracé unique final est fait par trace_temps_reel.
        if temps_relatif >= last_update_time + interval:
            t.append(temps_relatif)
            u.append(tension)
            last_update_time += interval
            if affichage_temps_reel:
                # plotly redessine toute la trace à chaque mutation : en durée
                # infinie on borne la fenêtre affichée pour garder la fluidité
                # (le CSV plein débit, lui, ne perd rien). SEULE la donnée est
                # modifiée depuis ce thread ; l'échelle (autorange ou range
                # fixe) est posée sur le THREAD PRINCIPAL, cf. trace_temps_reel
                # et la cellule d'application — les mises à jour de layout
                # venues d'un thread de fond sont appliquées de manière
                # aléatoire par le frontend, d'où le défilement « erratique ».
                if duree_courante == float('inf'):
                    while len(t) > MAX_POINTS_VISIBLES and t[0] < t[-1] - FENETRE_AFFICHAGE_S:
                        t.pop(0)
                        u.pop(0)
                fig_graphique.data[0].x = list(t)
                fig_graphique.data[0].y = list(u)
        return True
    except Exception as e:
        print(f"Erreur lors de la mise à jour du graphique : {e}")
        return False


In [ ]:
import threading

# Graphique plotly : créé UNE SEULE FOIS et affiché par la cellule d'application
# (ArduinoTempsReel) ; chaque clic relance l'acquisition sur la même FigureWidget.
fig_graphique = None

# Arrêt de l'acquisition : levé par le bouton « Arrêter », testé par la boucle.
arret_demande = False

# Dernier message d'état de l'acquisition (affiché par la cellule
# d'application) : les print du thread de fond n'apparaissent pas ailleurs.
message_acquisition = ""

# Fenêtre d'affichage (durée infinie seulement) : plotly redessine toute la
# trace à chaque mise à jour ; au-delà de quelques centaines de points la
# cadence d'affichage souffrirait. On n'affiche donc que les 60 dernières
# secondes, le CSV plein débit gardant tout.
MAX_POINTS_VISIBLES = 600
FENETRE_AFFICHAGE_S = 60


def arreter_acquisition():
    """Demande l'arrêt de l'acquisition en cours (bouton « Arrêter »)."""
    global arret_demande
    arret_demande = True


def initialise_graphique():
    """Crée la FigureWidget plotly (une seule fois) et la renvoie.

    Une FigureWidget plotly s'affiche comme un widget ipywidgets : elle
    s'affiche avec display() et chaque mutation des données est transmise au
    canevas sans re-rendu d'image par le kernel. L'infobulle est native
    (hovertemplate), il n'y a pas de gadget invisible à ajouter."""
    global fig_graphique
    if fig_graphique is not None:
        return fig_graphique
    # Trace amorcée avec un point (0,0) : le premier rendu n'est JAMAIS une
    # scène vide, sinon plotly 6 (Figure + anywidget) affiche un canevas sans
    # données avec des ticks aberrants (lettres « b ») et le premier essai
    # donne l'impression d'une mesure ratée. Ce point est remplacé dès la
    # première mesure réelle.
    fig_graphique = go.FigureWidget(
        data=[go.Scatter(x=[0], y=[0], mode='lines', name='Tension',
                         hovertemplate='t = %{x:.3f} s<br>U = %{y:.3f} V<extra></extra>')],
        layout=dict(title='Tension en fonction du temps',
                    width=900, height=420,
                    margin=dict(l=45, r=20, t=40, b=35),
                    xaxis=dict(title='Temps (s)', showgrid=True,
                               gridcolor='#dddddd'),
                    yaxis=dict(title='Tension (V)', range=[0, 5],
                               showgrid=True, gridcolor='#dddddd'),
                    hovermode='x unified'))
    return fig_graphique


def regle_abscisse(duree):
    """Échelle en abscisse : 0 → durée si elle est finie, sinon suit la mesure.

    range=None (mode infini) efface l'échelle fixe éventuellement posée par une
    acquisition finie précédente, sinon plotly continue de l'appliquer."""
    if duree == float('inf'):
        fig_graphique.update_xaxes(autorange=True, range=None)
    else:
        fig_graphique.update_xaxes(autorange=False, range=[0, duree])


def sauvegarder_csv(fichier='arduino.csv'):
    """Écrit TOUTES les mesures reçues (temps en s, tension en V) dans un CSV."""
    global message_acquisition
    if not t_brut:
        message_acquisition = "Aucune donnée à enregistrer : acquisition vide."
        print(message_acquisition)
        return
    with open(fichier, 'w') as f:
        f.write("t,U\n")
        for ti, ui in zip(t_brut, u_brut):
            f.write(f"{ti:.6f},{ui:.6f}\n")
    message_acquisition = f"Enregistré dans {fichier} ({len(t_brut)} points plein débit)."
    print(message_acquisition)


def trace_temps_reel(duree):
    global fig_graphique, t, u, t_brut, u_brut, temps_initial, last_update_time, interval, dernier_temps, arret_demande, message_acquisition, affichage_temps_reel, duree_courante
    if fig_graphique is None:
        initialise_graphique()
    duree_courante = duree
    arret_demande = False  # re-levé par le bouton « Arrêter » pour stopper
    ser.timeout = 0.1  # readline ne bloque pas : arrêt réactif
    # Repartir du point d'amorçage (0,0) plutôt que d'une trace vide :
    # la scène plotly garde toujours au moins un point (évite le canevas
    # vide au premier rendu). L'échelle est posée avant le lancement de ce
    # thread par la cellule d'application via regle_abscisse() : on ne
    # touche JAMAIS au layout depuis ce thread.
    fig_graphique.data[0].x = [0]
    fig_graphique.data[0].y = [0]

    ser.reset_input_buffer()
    t = []
    u = []
    t_brut = []  # plein débit (tous les échantillons reçus)
    u_brut = []
    temps_initial = None
    dernier_temps = 0
    # Temps réel (un point par intervalle, ~100 points) pour les durées ≥ 0,1 s.
    # Durées très courtes : le temps réel est inutile, on collecte tout
    # (interval = 0) et on trace tous les points d'un seul coup à la fin.
    affichage_temps_reel = duree == float('inf') or duree >= 0.1
    if duree == float('inf'):
        interval = 0.1
    elif duree >= 0.1:
        interval = duree / 100 if duree / 100 > 0.02 else 0.02    # max() serait numpy.max (from numpy import *) : interdit !
    else:
        interval = 0    # chaque échantillon reçu est gardé pour le tracé final unique
    last_update_time = -interval    # premier point tracé dès t = 0 (immédiat si interval = 0)

    # Boucle de préchauffage pour lire et ignorer quelques lignes initiales
    for _ in range(10):
        ser.readline()

    silencieux = 0
    while True:
        if arret_demande:
            break  # stoppé via le bouton « Arrêter »
        if update_graph():
            silencieux = 0
        else:
            silencieux += 1
            if silencieux > 10:
                message_acquisition = "Aucune donnée reçue de l'Arduino : vérifier le câble USB."
                print(message_acquisition)
                break
        # Arrêt basé sur le temps de la dernière mesure reçue (indépendant de la
        # vitesse d'affichage) : durée réelle ≈ durée demandée.
        if duree != float('inf') and dernier_temps >= duree:
            break

    if not affichage_temps_reel and t:
        # Durée très courte : tracé unique, tous les points d'un coup
        fig_graphique.data[0].x = list(t)
        fig_graphique.data[0].y = list(u)
        regle_abscisse(duree)

    sauvegarder_csv()

    # Le port reste ouvert pour permettre une nouvelle acquisition d'un clic ;
    # le refermer en fin de session avec ser.close().
